In [1]:
# Get and extract the pdf file we want to chat with.

import os
import requests

remote_pdf_url = "https://arxiv.org/pdf/1709.00666.pdf"
pdf_filename = "ch02-downloaded.pdf"

if not os.path.exists(pdf_filename):
    response = requests.get(remote_pdf_url)

    if response.status_code == 200:
        with open(pdf_filename, "wb") as pdf_file:
            pdf_file.write(response.content)
    else:
        print("Failed to download the PDF. Status code:", response.status_code)
else:
    print(f"{pdf_filename} already exists, skipping download.")

In [2]:
import pdfplumber

text = ""
with pdfplumber.open(pdf_filename) as pdf:
    for page in pdf.pages:
        text += page.extract_text()

In [3]:
# Save the text as text file
with open("extracted_text.md", "w", encoding="utf-8") as md_file:
    md_file.write(text)

In [4]:
# Load the text here. 
# Later, the code can be run from here. No need to extract the pdf again.
with open("extracted_text.md", "r", encoding="utf-8") as md_file:
    loaded_text = md_file.read()

In [7]:
from utils import chunk_text

chunks = chunk_text(loaded_text, 500, 40)
print(len(chunks))

89


In [18]:
# now we embed the chunks

from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv(override=True)  # loads .env into environment variables

# Use OpenRouter API key and base URL
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)

def embed(texts, model="openai/text-embedding-3-small"):
    response = client.embeddings.create(
        input=texts,
        model=model,
    )
    return list(map(lambda n: n.embedding, response.data))

In [19]:
# Connect with Neo4j. I am using local

from neo4j import GraphDatabase
from openai import OpenAI
load_dotenv(override=True) 

driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"), os.getenv("NEO4J_PASSWORD"))
)

existing_records, _, _ = driver.execute_query(
    "MATCH (c:Chunk) WHERE c.embedding IS NOT NULL RETURN c.index AS idx, c.embedding AS embedding"
)
existing_embeddings = {record["idx"]: record["embedding"] for record in existing_records}

embeddings = [existing_embeddings.get(i) for i in range(len(chunks))]
missing_indexes = [i for i, emb in enumerate(embeddings) if emb is None]

if missing_indexes:
    missing_texts = [chunks[i] for i in missing_indexes]
    new_embeddings = embed(missing_texts)
    for idx, emb in zip(missing_indexes, new_embeddings):
        embeddings[idx] = emb
        driver.execute_query(
            "MATCH (c:Chunk {index: $idx}) SET c.embedding = $embedding",
            idx=idx,
            embedding=emb,
        )

driver.execute_query("""CREATE VECTOR INDEX pdf IF NOT EXISTS
FOR (c:Chunk) 
ON c.embedding""")

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `index` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=1, column=56, offset=55>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 55, 'line': 1, 'column': 56}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: 'MATCH (c:Chunk) WHERE c.embedding IS NOT NULL RETURN c.index AS idx, c.embedding AS embedding'


EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x000002ACC3835E50>, keys=[])

In [20]:
# Add to neo4j
cypher_query = '''
WITH $chunks as chunks, range(0, size($chunks)) AS index
UNWIND index AS i
WITH i, chunks[i] AS chunk, $embeddings[i] AS embedding
MERGE (c:Chunk {index: i})
SET c.text = chunk, c.embedding = embedding
'''

In [21]:
driver.execute_query(cypher_query, chunks=chunks, embeddings=embeddings)

EagerResult(records=[], summary=<neo4j._work.summary.ResultSummary object at 0x000002ACC74D6C90>, keys=[])

In [22]:
records, _, _ = driver.execute_query("MATCH (c:Chunk) WHERE c.index = 0 RETURN c.embedding, c.text")

print(records[0]["c.text"][0:30])
print(records[0]["c.embedding"][0:3])

Einstein’s Patents and Inventi
[0.0237579345703125, -0.0224761962890625, -0.0145263671875]


In [23]:
question = "At what time was Einstein really interested in experimental works?"
question_embedding = embed([question])[0]

query = '''
CALL db.index.vector.queryNodes('pdf', $k, $question_embedding) YIELD node AS hits, score
RETURN hits.text AS text, score, hits.index AS index
'''
similar_records, _, _ = driver.execute_query(query, question_embedding=question_embedding, k=4)

for record in similar_records:
    print(record["text"])
    print(record["score"], record["index"])
    print("======")

CH‐Switzerland
Considering Einstein’s upbringing, his interest in inventions and patents was not unusual.
Being a manufacturer’s son, Einstein grew upon in an environment of machines and instruments.
When his father’s company obtained the contract to illuminate Munich city during beer festival, he
was actively engaged in execution of the contract. In his ETH days Einstein was genuinely interested
in experimental works. He wrote to his friend, “most of the time I worked in the physical laboratory,
fascinated by the direct contact with observation.” Einstein's
0.8110485076904297 42
Einstein
left his job at the Patent office and joined the University of Zurich on October 15, 1909. Thereafter, he
continued to rise in ladder. In 1911, he moved to Prague University as a full professor, a year later, he
was appointed as full professor at ETH, Zurich, his alma‐mater. In 1914, he was appointed Director of
the Kaiser Wilhelm Institute for Physics (1914–1932) and a professor at the Humboldt Unive

In [24]:
system_message = "You're en Einstein expert, but can only use the provided documents to respond to the questions."
user_message = f"""
Use the following documents to answer the question that will follow:
{[doc["text"] for doc in similar_records]}

---

The question to answer using information only from the above documents: {question}
"""

print("Question:", question)

stream = client.chat.completions.create(
    model="gpt-4",
    messages=[
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message}
    ],
    stream=True,
)
for chunk in stream:
    print(chunk.choices[0].delta.content or "", end="")

Question: At what time was Einstein really interested in experimental works?
Einstein was genuinely interested in experimental works during his ETH days, according to the documents.